# trajectory-kit: Tutorial

Welcome to `trajectory-kit` — a small library for loading molecular dynamics
files and answering questions about them.

You'll learn to:

- Load a simulation from typing, topology, and trajectory files
- Write **queries** (e.g. "all CA atoms in the protein segment")
- Use the three user-facing methods:
  - `positions()` — coordinate arrays for selected atoms
  - `fetch()` — per-atom property lists for selected atoms
  - `select()` — file-wide scalar properties

This tutorial keeps things simple and stays at the surface. When you're ready
for more:

- **`tutorial_advanced.ipynb`** opens up the full result envelope, the
  planner, and cost estimation.
- **`tutorial_dev.ipynb`** covers internals — iterators, the parser contract,
  and how to add a new file format.

This tutorial uses a 20-atom synthetic system (ALA-GLY-PRO backbone plus two
TIP3 waters). The same data is used across all three tutorials so concepts
carry over.


## 0. Setup — synthetic data files

We'll generate a small PDB (typing), PSF (topology), and DCD (trajectory)
file in a temp directory. The path is printed at the bottom of the cell —
you can browse there to inspect the raw files if you're curious.


In [1]:
import struct
import tempfile
from pathlib import Path
import numpy as np

# Synthetic files are written to a fresh temp directory. The path is printed
# below so you can browse to it and inspect the raw files if you want to.
tmp = Path(tempfile.mkdtemp(prefix="trajkit_tutorial_"))

# ── 20-atom synthetic system ──────────────────────────────────────────────
# Protein (PROT): ALA(6) + GLY(5) + PRO(3) = 14 atoms, 3 residues
# Solvent (SOLV): 2 x TIP3 water molecules = 6 atoms, 2 residues
#
# Bond graph (17 bonds):
#   Backbone: N-CA-C chain with peptide bonds
#   ALA sidechain: CA-CB
#   Carbonyl: C=O on each residue
#   PRO N closes the chain
#   Waters: each OH2 bonded to H1 and H2
#
# Degree distribution: 10 x degree-1, 6 x degree-2, 4 x degree-3
# Atom types used: NH1, H, CT1, CT3, C, O, CT2, N, CP1, OT, HT

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

psf_text = """\
PSF

       1 !NTITLE
 REMARKS synthetic 20-atom ALA-GLY-PRO + 2xTIP3

      20 !NATOM
       1 PROT     1 ALA  N    NH1     -0.47000     14.007           0
       2 PROT     1 ALA  HN   H        0.31000      1.008           0
       3 PROT     1 ALA  CA   CT1     -0.02000     12.011           0
       4 PROT     1 ALA  CB   CT3     -0.27000     12.011           0
       5 PROT     1 ALA  C    C        0.51000     12.011           0
       6 PROT     1 ALA  O    O       -0.51000     15.999           0
       7 PROT     2 GLY  N    NH1     -0.47000     14.007           0
       8 PROT     2 GLY  HN   H        0.31000      1.008           0
       9 PROT     2 GLY  CA   CT2     -0.18000     12.011           0
      10 PROT     2 GLY  C    C        0.51000     12.011           0
      11 PROT     2 GLY  O    O       -0.51000     15.999           0
      12 PROT     3 PRO  N    N       -0.29000     14.007           0
      13 PROT     3 PRO  CA   CP1     -0.02000     12.011           0
      14 PROT     3 PRO  C    C        0.51000     12.011           0
      15 SOLV     4 TIP  OH2  OT      -0.83400     15.999           0
      16 SOLV     4 TIP  H1   HT       0.41700      1.008           0
      17 SOLV     4 TIP  H2   HT       0.41700      1.008           0
      18 SOLV     5 TIP  OH2  OT      -0.83400     15.999           0
      19 SOLV     5 TIP  H1   HT       0.41700      1.008           0
      20 SOLV     5 TIP  H2   HT       0.41700      1.008           0

      17 !NBOND: bonds
       1       2       1       3       3       4       3       5
       5       6       5       7       7       8       7       9
       9      10      10      11      10      12      12      13
      13      14      15      16      15      17      18      19
      18      20

       0 !NTHETA: angles

       0 !NPHI: dihedrals

       0 !NIMPHI: impropers

"""

def write_dcd(path, n_atoms=20, n_frames=5):
    """5-frame DCD. Each frame shifts all atoms by +1 A along x."""
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20
    icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synthetic.pdb'
psf_path = tmp / 'synthetic.psf'
dcd_path = tmp / 'synthetic.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
psf_path.write_text(psf_text, encoding='ascii')
write_dcd(dcd_path)

print('Synthetic files written to:', tmp)
print('  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)')
print('  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)')
print('  DCD: 5 frames - x coordinates shift +1 A per frame')


Synthetic files written to: /var/folders/b7/cnz6gkn92hl85yrrz4d00lgm0000gn/T/trajkit_tutorial_8z_mnrze
  PDB: 20 atoms (ALA-GLY-PRO backbone + 2x TIP3 water)
  PSF: 20 atoms, 17 bonds, 5 residues, 2 segments (PROT/SOLV)
  DCD: 5 frames - x coordinates shift +1 A per frame


## 1. Loading a sim

A `sim` object is the entry point. Point it at any combination of files —
the three roles are:

- **Typing** (`.pdb`, `.xyz`, `.mae`) — the atom roster: which atoms exist,
  what each is called, what residue/segment it belongs to. Static.
- **Topology** (`.psf`, `.mae`) — the bond graph and per-atom chemistry
  (charges, masses, atom types). Static.
- **Trajectory** (`.dcd`, `.coor`) — atomic positions, sampled over time.
  May be one frame (`.coor`) or many (`.dcd`).

You can load one, two, or all three. Anything not loaded simply isn't queryable.


In [2]:
from trajectory_kit import sim

s = sim(
    typing=pdb_path,
    topology=psf_path,
    trajectory=dcd_path,
)

print('Loaded:', type(s).__name__)


Loaded: sim


## 2. The query schema

A query is a plain Python dictionary. Each key names a queryable field; each
value tells `sim` which atoms to keep or drop.

There are two kinds of fields:

**Categorical fields** — string-valued (atom names, residue names, segment
names). Match against an *include set* and an *exclude set*.

**Range fields** — numeric (residue ids, coordinates, occupancy, charges).
Match against include/exclude ranges. The container type is the
disambiguation signal:

- **tuple** `(low, high)` → an inclusive range. `(None, 100)` means "up to
  100"; `(5, None)` means "5 and above".
- **list** `[v1, v2, ...]` → exact membership of those specific values.

So for residue ids, `(1, 5)` means residues 1 *through* 5, while `[1, 5]`
means residues 1 *and* 5 only.

For both kinds (categorical and range) you may supply just the value
(treated as include only) or the explicit `(include, exclude)` tuple.

| Style | Example | Meaning |
|-------|---------|---------|
| Bare value (categorical) | `{'atom_name': 'CA'}` | include only `CA` atoms |
| Set of values | `{'atom_name': {'CA', 'CB'}}` | include `CA` or `CB` |
| Include + exclude (categorical) | `{'atom_name': ({'CA'}, {'HN'})}` | include `CA`, exclude `HN` |
| Range tuple | `{'residue_ids': (1, 5)}` | include residues 1 through 5 |
| Membership list | `{'residue_ids': [1, 5]}` | include residues 1 and 5 only |
| Open range | `{'residue_ids': (None, 100)}` | up to residue 100 |
| Include + exclude (range) | `{'residue_ids': ((1, 10), [3, 7])}` | include 1-10, exclude 3 and 7 |

**Datatypes:** include and exclude *sets* should be Python `set` or `list`
(either works — they're normalised internally). An empty query `{}` means
"all atoms".


### Glossary — the keywords you'll see

These names show up in queries and in the `REQUEST` strings you'll meet
shortly.

**From the typing file (PDB):**

| Keyword | Meaning |
|---------|---------|
| `global_ids` | Zero-based atom index across the whole system. The canonical handle. |
| `local_ids` | The atom serial number as written in the file (often 1-based). |
| `atom_name` | The atom label (e.g. `CA`, `HN`, `OH2`). |
| `residue_ids` | Numeric residue index. |
| `residue_name` | Residue label (e.g. `ALA`, `GLY`, `TIP`). |
| `segment_name` | Segment label (e.g. `PROT`, `SOLV`). |
| `x`, `y`, `z` | Atomic coordinates (numeric). |
| `occupancy` | PDB occupancy column (numeric). |
| `temperature_coeff` | PDB B-factor column (numeric). Used in crystallography for atomic disorder; carried through here mainly because the PDB column exists. |

**From the topology file (PSF):**

| Keyword | Meaning |
|---------|---------|
| `atom_type` | Force-field atom type (e.g. `NH1`, `CT3`). |
| `charge` | Partial charge (numeric range). |
| `mass` | Atomic mass (numeric range). |
| `bonded_with` | Bond-graph constraint — see Section 7. |

**Trajectory queries** take pre-resolved `global_ids` and an optional
`frame_interval` (a `(start, stop, step)` tuple).


## 3. Two ways to ask questions

`trajectory-kit` separates questions about **atoms** from questions about
the **whole system**:

- For data attached to atoms (positions, names, charges) use **`positions()`**
  or **`fetch()`**. You give a query that selects atoms; you get back data
  for those atoms.
- For data about the whole file (atom count, box size) use **`select()`**.
  No selection needed — these are scalar properties of the system.

Both produce results without needing to manually combine domains: when you
query across typing and topology together, `positions()` and `fetch()`
automatically intersect the selections so every value you get back belongs
to the same set of atoms.


## 4. Getting positions

`positions()` returns a NumPy array of shape `(n_frames, n_atoms_selected, 3)`.
With no trajectory loaded the shape is `(1, n_atoms_selected, 3)` — a single
frame from the static file.


In [3]:
# All CA atoms across all 5 frames.
ca_pos = s.positions(TYPE_Q={'atom_name': 'CA'})

print('shape:', ca_pos.shape)              # (5, 3, 3) — 5 frames, 3 CAs, xyz
print('first frame CA positions:')
print(ca_pos[0])


shape: (5, 3, 3)
first frame CA positions:
[[2.4 5.  5. ]
 [5.8 3.2 4.1]
 [9.5 2.8 3.6]]


The trajectory query supports `frame_interval = (start, stop, step)` to read
a subset of frames. Bounds are half-open: `start` is inclusive, `stop` is
exclusive. Default is all frames.


In [4]:
# CA atoms, every other frame from frame 0 to (not including) 4.
ca_skip = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TRAJ_Q={'frame_interval': (0, 4, 2)},
)

print('shape:', ca_skip.shape)             # (2, 3, 3)
print(ca_skip)


shape: (3, 3, 3)
[[[ 2.4  5.   5. ]
  [ 5.8  3.2  4.1]
  [ 9.5  2.8  3.6]]

 [[ 4.4  5.   5. ]
  [ 7.8  3.2  4.1]
  [11.5  2.8  3.6]]

 [[ 6.4  5.   5. ]
  [ 9.8  3.2  4.1]
  [13.5  2.8  3.6]]]


### Cross-domain selection

When you supply both `TYPE_Q` and `TOPO_Q`, the selections are
**intersected**. The returned positions are the atoms matching *both*
queries.


In [5]:
# Typing: CA atoms                  → atoms 3, 9, 13   (1-based local_ids)
# Topology: PROT segment via charge → all PROT atoms
# Intersection: PROT CAs only.
prot_cas = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TOPO_Q={'charge': (-1.0, 1.0)},   # all PROT atoms have small charges
)

print('shape:', prot_cas.shape)
print('first frame:')
print(prot_cas[0])


shape: (5, 3, 3)
first frame:
[[2.4 5.  5. ]
 [5.8 3.2 4.1]
 [9.5 2.8 3.6]]


## 5. Fetching per-atom properties

`fetch()` is the general per-atom extractor. You give it a query and a
**request string** per domain. It returns a dict keyed by domain.

The request string names what you want back — `'atom_names'`, `'charges'`,
`'positions'`, etc. (Section 8 below shows how to discover what's available
for the loaded files.)


In [6]:
# Get residue names for CA atoms.
result = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TYPE_R='residue_names',
)

print(result)
# {'typing': ['ALA', 'GLY', 'PRO']}


{'typing': ['ALA', 'GLY', 'PRO']}


You can ask for things from multiple domains in a single call. Cross-domain
intersection still applies: every list comes back with values for the
*same* atoms.


In [7]:
# CA atoms - get their residue names AND their PSF charges in one call.
result = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TYPE_R='residue_names',
    TOPO_R='charges',
)

print('residue_names :', result['typing'])
print('charges       :', result['topology'])


residue_names : ['ALA', 'GLY', 'PRO']
charges       : [-0.02, -0.18, -0.02]


## 6. Selecting global properties

Properties of the whole file (not tied to specific atoms) start with
`property-`. Use `select()` for these. The result is a dict keyed by domain;
each value is a scalar or tuple.


In [8]:
n_atoms    = s.select(TYPE_R='property-number_of_atoms')
n_residues = s.select(TYPE_R='property-number_of_residues')
n_segments = s.select(TYPE_R='property-number_of_segments')
box_size   = s.select(TYPE_R='property-box_size')

print('atoms    :', n_atoms)        # {'typing': 20}
print('residues :', n_residues)     # {'typing': 5}
print('segments :', n_segments)     # {'typing': 2}
print('box      :', box_size)       # {'typing': (xmin, xmax, ymin, ymax, zmin, zmax)}


atoms    : {'typing': 20}
residues : {'typing': 5}
segments : {'typing': 2}
box      : {'typing': (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)}


## 7. Bond-graph queries

The topology file carries a bond graph, and you can select atoms by their
chemical neighbourhood using the `bonded_with` keyword.

A `bonded_with` value describes one or more **neighbour requirements**.
Each requirement is a dict with:

- `count` — a comparator on the number of matching neighbours, e.g.
  `{'eq': 1}`, `{'ge': 2}`, `{'lt': 4}`. Operators: `eq`, `ne`, `ge`,
  `le`, `gt`, `lt`.
- Either `total: True` (count *all* neighbours) **or** `neighbor: {...}`
  (a sub-query selecting which neighbours to count).

The `neighbor` sub-query uses the same query schema as a top-level query,
so any keyword you can write in `TOPO_Q` you can write inside `neighbor`.

### Shorthand for `bonded_with`

For the common single-block case you can pass the dict directly. For the
common multi-block-include case you can pass a list. The full form
(`(include_blocks, exclude_blocks)`) stays valid for queries that need both.

| Form | Use |
|------|-----|
| `bonded_with: {'count': {'eq': 1}, 'total': True}` | one include block (most common) |
| `bonded_with: [{...}, {...}]` | several include blocks |
| `bonded_with: ([...], [...])` | full include + exclude |

Bond filtering applies uniformly to **every** topology request — `global_ids`,
`atom_names`, `charges`, etc. — so you can ask for any field of the
bond-filtered atom set in one call.


In [9]:
# Terminal atoms (degree 1) - HNs, CB, terminal Os, water hydrogens, PRO C.
# Bare-dict shorthand for a single bonded_with block.
terminal_names = s.fetch(
    TOPO_Q={'bonded_with': {'total': True, 'count': {'eq': 1}}},
    TOPO_R='atom_names',
)
print('Terminal atoms:', terminal_names['topology'])

# Atoms bonded to a water oxygen (OT) - the water hydrogens.
# Notice the inner 'atom_type': 'OT' uses the bare-string shorthand too.
near_water = s.fetch(
    TOPO_Q={'bonded_with': {'neighbor': {'atom_type': 'OT'}, 'count': {'ge': 1}}},
    TOPO_R='atom_names',
)
print('Bonded to water O:', near_water['topology'])

# Charges of those same atoms - bond filter applies uniformly to charges too.
near_water_charges = s.fetch(
    TOPO_Q={'bonded_with': {'neighbor': {'atom_type': 'OT'}, 'count': {'ge': 1}}},
    TOPO_R='charges',
)
print('Charges of those atoms:', near_water_charges['topology'])


Terminal atoms: ['HN', 'CB', 'O', 'HN', 'O', 'C', 'H1', 'H2', 'H1', 'H2']
Bonded to water O: ['H1', 'H2', 'H1', 'H2']
Charges of those atoms: [0.417, 0.417, 0.417, 0.417]


## 8. Discovering what's available

Use `print_info()` for a human-readable overview of the loaded files and
everything you can query or request from them.


In [10]:
s.print_info()



=== SIMULATION INFO ===

  files
    typing     synthetic.pdb  (.pdb)
    topology   synthetic.psf  (.psf)
    trajectory synthetic.dcd  (.dcd)

  system properties
    num_atoms        20
    num_residues     5
    num_frames       5
    start_box_size   (0.1, 14.8, 1.9, 6.3, 3.2, 5.5)

  available keywords and requests
  ──────────  ───────────────────────────────  ──────────────────────────  ────────────────
              typing                           topology                    trajectory      
  ──────────  ───────────────────────────────  ──────────────────────────  ────────────────
  keywords    atom_name                        atom_name                   frame_interval  
              global_ids                       atom_type                   global_ids      
              local_ids                        bonded_with                                 
              occupancy                        bonded_with_mode                            
              residue_ids       

## 9. Static use — no trajectory needed

You don't have to load a trajectory file. With just a typing or topology
file, `positions()` returns the single static frame from that file (shape
`(1, n_atoms, 3)`), and `fetch()` / `select()` work as usual.


In [11]:
# Typing-only sim - works fine.
s_static = sim(typing=pdb_path)

static_pos = s_static.positions(TYPE_Q={'atom_name': 'CA'})
print('static shape:', static_pos.shape)   # (1, 3, 3)

ca_resnames = s_static.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TYPE_R='residue_names',
)
print('CA resnames :', ca_resnames['typing'])


static shape: (1, 3, 3)
CA resnames : ['ALA', 'GLY', 'PRO']


## What next?

That's the whole user-facing surface. With these three methods you can write
practical analyses without ever opening a parser file.

When you want more:

- **`tutorial_advanced.ipynb`** introduces `devFlag=True`, which switches
  every method to return a structured envelope with selection details, file
  metadata, and a planner that estimates payload size *before* you read.
- **`tutorial_dev.ipynb`** walks through the parser contract, the iterator
  helpers, and how to add a new file format.
